## **Setup**

### **Import Packages**

In [ ]:
%%capture
%pip install sdmetrics numpy pandas matplotlib sdv

In [ ]:
!git clone https://github.com/SwanseaUniversityMedical/syntheticdata-masterclass.git /content/shared-workshop-group-1

In [ ]:
import json
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
project_root = "/content/shared-workshop-group-1"
sys.path.append(project_root)
from AusSynthPackage import process, generate, evaluate, metadata_to_html, generate_html_report

### **Read in the Source Data**

Read the three tables into dataframes, and view the head of each table to see how they look.

In [ ]:
DATA = "/content/shared-workshop-group-1/example_data" # path to the directory containing the CSV files

patients = pd.read_csv(f"{DATA}/patients.csv")
conditions = pd.read_csv(f"{DATA}/conditions.csv")
medications = pd.read_csv(f"{DATA}/medications.csv")

##### ***Task**: Look at the heads of the three different dataframes to get an idea of how they look*

In [ ]:
patients.head()

##### ***Task**: Identify how many rows and columns each of the tables have*

In [ ]:
patients.shape, conditions.shape, medications.shape

##### ***Task**: Visualise Variable Distributions*

In [ ]:
# TASK Plot distribution of a variable from one of the tables

plt.hist(patients['RACE'])  # Replace 'VARIABLE_NAME' with an actual column name
plt.show()

### **Relational Table Setup**

The data we will be using in this exercise contains three different tables: patients, conditions and medications. The patients table has a one-to-many relationship with conditions and with medications, where for each patient record, there may be multiple conditions that patient has, or multiple medications that patient may have taken. 

Before we get started, we need to specify what the primary and foreign keys are for our data, and how they relate.

##### ***Task**: Specify Primary and Foreign Keys*

In [ ]:
# TASK: Specify primary key for the parent table and foreign keys for the child tables

primary_keys = {
    "TABLE_NAME": "COLUMN_NAME",
}

foreign_keys = {
    "TABLE_NAME": [
        {
            "column": "FOREIGN_KEY_COLUMN_NAME",
            "references_table": "PARENT_TABLE_NAME",
            "references_column": "PARENT_PRIMARY_KEY_COLUMN_NAME"
        }
    ],
    "TABLE_NAME": [
        {
            "column": "FOREIGN_KEY_COLUMN_NAME",
            "references_table": "PARENT_TABLE_NAME",
            "references_column": "PARENT_PRIMARY_KEY_COLUMN_NAME"
        }
    ]
}

##### Specify Linked Column Pairs

There are some variables in the tables which represent the same thing, but just in a different format. For example code is the numerical representation, and description is the string representation. We need to specify this logic so the generation process understands this link.

In [ ]:
linked_columns = {
    "conditions": [
        ["CODE", "DESCRIPTION"]
    ],
    "medications": [
        ["CODE", "DESCRIPTION"],
        ["REASONCODE", "REASONDESCRIPTION"]
    ],
}

## **Process Metadata**

#### **Create Metadata**

In [ ]:
metadata = process(
    {
        "patients": f"{DATA}/patients.csv",
        "conditions": f"{DATA}/conditions.csv",
        "medications": f"{DATA}/medications.csv",
    },
    sdc_threshold=10, # Statistical Disclosure Control threshold for supressing low-frequency values
    n_bins=20, # Number of bins to use for numerical continuous columns when calculating value counts
    primary_keys=primary_keys,
    foreign_keys=foreign_keys,
    linked_columns=linked_columns,
 )

#### **Inspect the Metadata**

In [ ]:
metadata_to_html(metadata, output_path="metadata.html");

In [ ]:
from IPython.display import HTML, display

with open('/content/metadata.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))

## **Generate Level 1 Synthetic Data**

#### **Create Level 1 Synthetic Data**

In [ ]:
synthetic_data_level_1 = generate(metadata, n_rows={"patients": 1000}, level=1)

#### **Inspect Level 1 Synthetic Data Dataframe**

In [ ]:
synthetic_data_level_1['patients'].head()

#### **Visualise & Compare Distributions**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

patients["RACE"].value_counts().plot(kind="bar", ax=axes[0], color="blue")
axes[0].set_title("Original Data")
axes[0].set_xlabel("RACE")

synthetic_data_level_1["patients"]["RACE"].value_counts().plot(kind="bar", ax=axes[1], color="red")
axes[1].set_title("Synthetic Data")
axes[1].set_xlabel("RACE")

plt.tight_layout()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the distributions*

In [ ]:
# TASK: PLOT DISTRIBUTION OF ANOTHER VARIABLE

#### **Visualise & Compare Boxplots**

In [ ]:
plt.figure(figsize=(10, 6))
patients.boxplot(column="INCOME", by="RACE", grid=False, showfliers=False)
plt.xlabel("RACE")
plt.ylabel("INCOME")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
synthetic_data_level_1['patients'].boxplot(column="INCOME", by="RACE", grid=False, showfliers=False)
plt.xlabel("RACE")
plt.ylabel("INCOME")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the boxplots*

In [ ]:
# TASK: PLOT BOXPLOTS OF OTHER VARIABLES

#### **Visualise & Compare Correlations**

In [ ]:
plt.scatter(patients["LON"], patients["LAT"], label="Original Data", color='blue', alpha=0.3)
plt.scatter(synthetic_data_level_1['patients']["LON"], synthetic_data_level_1['patients']["LAT"], label="Synthetic Data", color='red', alpha=0.3)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the scatterplots*

In [ ]:
# TASK: PLOT SCATTERPLOTS OF OTHER VARIABLES

## **Generate Level 2 Synthetic Data**

#### **Create Level 2 Synthetic Data**

In [ ]:
synthetic_data_level_2 = generate(metadata, n_rows={"patients": 1000}, level=2)

#### **Visualise & Compare Distributions**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

patients["RACE"].value_counts().plot(kind="bar", ax=axes[0], color="blue")
axes[0].set_title("Original Data")
axes[0].set_xlabel("RACE")

synthetic_data_level_2["patients"]["RACE"].value_counts().plot(kind="bar", ax=axes[1], color="red")
axes[1].set_title("Synthetic Data")
axes[1].set_xlabel("RACE")

plt.tight_layout()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the distributions*

In [ ]:
# TASK: PLOT DISTRIBUTION OF ANOTHER VARIABLE

#### **Visualise & Compare Boxplots**

In [ ]:
plt.figure(figsize=(10, 6))
patients.boxplot(column="INCOME", by="RACE", grid=False, showfliers=False)
plt.xlabel("RACE")
plt.ylabel("INCOME")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
synthetic_data_level_2['patients'].boxplot(column="INCOME", by="RACE", grid=False, showfliers=False)
plt.xlabel("RACE")
plt.ylabel("INCOME")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the boxplots*

In [ ]:
# TASK: PLOT BOXPLOTS OF OTHER VARIABLES

#### **Visualise & Compare Correlations**

In [ ]:
plt.scatter(patients["LON"], patients["LAT"], label="Original Data", color='blue', alpha=0.3)
plt.scatter(synthetic_data_level_2['patients']["LON"], synthetic_data_level_2['patients']["LAT"], label="Synthetic Data", color='red', alpha=0.3)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the scatterplots*

In [ ]:
# TASK: PLOT SCATTERPLOTS OF OTHER VARIABLES

## **Generate Level 3 Synthetic Data**

#### **Create Level 3 Metadata**

In [ ]:
metadata_level_3 = process(
    {
        "patients":    f"{DATA}/patients.csv",
        "conditions":  f"{DATA}/conditions.csv",
        "medications": f"{DATA}/medications.csv",
    },
    sdc_threshold=10,
    n_bins=20,
    primary_keys=primary_keys,
    foreign_keys=foreign_keys,
    linked_columns=linked_columns,
    level=3,
    n_parent_context_cols=patients.shape[1],
)

#### **Create Level 3 Synthetic Data**

In [ ]:
synthetic_data_level_3 = generate(metadata_level_3, n_rows={"patients": 1000}, level=3)

#### **Visualise & Compare Distributions**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

patients["RACE"].value_counts().plot(kind="bar", ax=axes[0], color="blue")
axes[0].set_title("Original Data")
axes[0].set_xlabel("RACE")

synthetic_data_level_3["patients"]["RACE"].value_counts().plot(kind="bar", ax=axes[1], color="red")
axes[1].set_title("Synthetic Data")
axes[1].set_xlabel("RACE")

plt.tight_layout()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the distributions*

In [ ]:
# TASK: PLOT DISTRIBUTION OF ANOTHER VARIABLE

#### **Visualise & Compare Boxplots**

In [ ]:
plt.figure(figsize=(10, 6))
patients.boxplot(column="INCOME", by="RACE", grid=False, showfliers=False)
plt.xlabel("RACE")
plt.ylabel("INCOME")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
synthetic_data_level_3['patients'].boxplot(column="INCOME", by="RACE", grid=False, showfliers=False)
plt.xlabel("RACE")
plt.ylabel("INCOME")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the boxplots*

In [ ]:
# TASK: PLOT BOXPLOTS OF OTHER VARIABLES

#### **Visualise & Compare Correlations**

In [ ]:
plt.scatter(patients["LON"], patients["LAT"], label="Original Data", color='blue', alpha=0.3)
plt.scatter(synthetic_data_level_3['patients']["LON"], synthetic_data_level_3['patients']["LAT"], label="Synthetic Data", color='red', alpha=0.3)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

##### ***Task**: Have a go at plotting different variables and look at the scatterplots*

In [ ]:
# TASK: PLOT SCATTERPLOTS OF OTHER VARIABLES

## **Evaluate Synthetic Data**

In [ ]:
real_tables = {
    "patients":f"{DATA}/patients.csv",
    "conditions":f"{DATA}/conditions.csv",
    "medications":f"{DATA}/medications.csv"
    }

#### **Evaluate Level 1 Synthetic Data**

In [ ]:
results = evaluate(
    real_tables=real_tables,
    synthetic_tables=synthetic_data_level_1,
    metadata=metadata,
)

##### Create Synthetic Data Level 1 HTML Report

In [ ]:
plot_spec = {
    "categorical_distributions": [
        ("patients", "RACE"),
        ("medications", "DISPENSES"),
    ],
    "continuous_distributions": [
        ("patients", "INCOME"),
        ("patients", "HEALTHCARE_EXPENSES"),
    ],
    "boxplots": [
        ("patients", "RACE", "INCOME"),
        ("patients", "COUNTY", "HEALTHCARE_EXPENSES"),
    ],
    "scatter_plots": [
        ("patients", "LON", "LAT"),
        ("patients", "HEALTHCARE_EXPENSES", "INCOME"),
    ],
    "correlation_tables": [
        "patients",
        "medications",
    ],
    "cardinality_relationships": [
        ("patients", "conditions"),
    ],
}

generate_html_report(
    real_tables=real_tables,
    synthetic_tables=synthetic_data_level_1,
    metadata=metadata,
    output_path=f"/content/ausynth_report_level_1.html",
    plot_spec=plot_spec,
);

In [ ]:
from IPython.display import HTML, display

with open('/content/ausynth_report_level_1.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))

##### ***TASK:** Create Synthetic Data Level 2 and Level 3 HTML Report*

In [ ]:

#TASK: GENERATE REPORT FOR LEVEL 2 AND LEVEL 3 SYNTHETIC DATA

## **Adjust Synthetic Data Parameters and Investigate Effect on Results**

Have a go at changing some parameters in Level 3 synthetic data such as differential privacy, number of bins, SDC threshold etc and see how it affects the quality, utility and privacy scores.

In [ ]:
# TASK: ADJUST SYNTHESIS PARAMETERS AND EVALUATE IMPACT ON SYNTHETIC DATA RESULTS

metadata_level_3_adjusted = process(
    {
        "patients":    f"{DATA}/patients.csv",
        "conditions":  f"{DATA}/conditions.csv",
        "medications": f"{DATA}/medications.csv",
    },
    sdc_threshold=10,                   # Try adjusting this threshold
    n_bins=20,                          # Try adjusting the number of bins for numerical columns
    primary_keys=primary_keys,
    foreign_keys=foreign_keys,
    linked_columns=linked_columns,
    level=3,
    n_parent_context_cols=patients.shape[1],
)

synthetic_data_level_3_adjusted = generate(
    metadata_level_3_adjusted, 
    n_rows={"patients": 1000}, 
    level=3,
    dp_epsilon=0.01,                    # Try adjusting the amount of differential privacy noise added
)

In [ ]:
# TASK: Generate Report for Adjusted Level 3 Synthetic Data

generate_html_report(
    real_tables=real_tables,
    synthetic_tables=synthetic_data_level_3_adjusted,
    metadata=metadata,
    output_path=f"/content/ausynth_report_level_3_adjusted.html",
    plot_spec=plot_spec,
);

In [ ]:
from IPython.display import HTML, display

with open('/content/ausynth_report_level_3_adjusted.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))